# Document Chunking Quality Benchmark

Evaluates 4 chunking strategies across AIF example documents.

**Why chunking quality matters for LLMs:** When documents are split into chunks for retrieval-augmented generation or multi-step reasoning, the quality of those chunks directly affects LLM performance. Self-contained chunks with clear semantic boundaries help LLMs:

- **Follow instructions more accurately** --- a chunk that begins with a section title and contains complete blocks gives the model clear context for what it is reading.
- **Avoid hallucination at chunk edges** --- poorly split chunks that break mid-paragraph or mid-block force the model to guess at missing context.
- **Respect typed structure** --- AIF's semantic blocks (`@claim`, `@evidence`, `@step`, `@verify`) carry meaning that should not be fragmented across chunks.

This benchmark is **fully deterministic** --- it uses the `aif chunk split` CLI command and requires no API key.

## Setup

In [ ]:
import subprocess
import json
import math
import os
import statistics
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath("")).parent.parent
print(f"Project root: {PROJECT_ROOT}")

## Strategies Overview

AIF supports 4 document chunking strategies, each with different trade-offs:

| Strategy | How it splits | Best for |
|----------|--------------|----------|
| **Section** | Splits on document section boundaries (`@section` blocks). Each section becomes its own chunk. | Documents with clear hierarchical structure --- each chunk maps to a logical topic. |
| **Token Budget** | Greedily fills chunks up to a target token limit (default 2048). Respects block boundaries. | Fitting chunks into LLM context windows --- guarantees no chunk exceeds the budget. |
| **Semantic** | Splits on semantic block boundaries (`@claim`, `@evidence`, `@step`, etc.). Groups related typed blocks together. | Preserving meaning --- keeps semantic units intact so LLMs can reason about typed content. |
| **Fixed Blocks** | Splits every N blocks (default ~5) regardless of content. | Uniform chunk sizes --- simple and predictable, but may cut across semantic boundaries. |

### Metrics

- **Self-Containment**: Fraction of chunks that have a title (a titled chunk gives the LLM immediate context about what it is reading).
- **Size CV** (Coefficient of Variation): How uniform chunk sizes are. Lower is better --- wildly varying chunk sizes waste context window space on tiny chunks and may overflow on large ones.
- **Avg Blocks/Chunk**: Average number of AIF blocks per chunk. More blocks per chunk means more context per retrieval hit.
- **Budget Compliance** (token-budget only): Fraction of chunks within 10% of the target token limit.

## Configuration

In [ ]:
STRATEGIES = ["section", "token-budget", "semantic", "fixed-blocks"]
MAX_TOKENS = 2048

# Discover all example .aif files
examples_dir = PROJECT_ROOT / "examples"
aif_files = sorted(examples_dir.glob("**/*.aif"))
print(f"Found {len(aif_files)} .aif files:")
for f in aif_files:
    print(f"  {f.relative_to(PROJECT_ROOT)}")

## Chunking Functions

In [ ]:
def run_cli(args: list[str]) -> str:
    """Run aif-cli and return stdout."""
    result = subprocess.run(
        ["cargo", "run", "-p", "aif-cli", "--"] + args,
        capture_output=True, text=True, cwd=str(PROJECT_ROOT),
    )
    if result.returncode != 0:
        print(f"  CLI error: {result.stderr.strip()[:200]}")
        return ""
    return result.stdout


def chunk_document(aif_file: str, strategy: str, max_tokens: int = MAX_TOKENS) -> list[dict]:
    """Chunk a document and return parsed chunk metadata."""
    args = ["chunk", "split", aif_file, "--strategy", strategy]
    if strategy == "token-budget":
        args += ["--max-tokens", str(max_tokens)]
    output = run_cli(args)
    if not output:
        return []

    chunks = []
    for line in output.strip().split("\n"):
        if not line.strip():
            continue
        # Format: "chunk_id | blocks: N | tokens: ~T | title: ..."
        parts = line.split("|")
        if len(parts) >= 4:
            chunk_id = parts[0].strip()
            blocks = int(parts[1].strip().replace("blocks: ", ""))
            tokens = int(parts[2].strip().replace("tokens: ~", ""))
            title = parts[3].strip().replace("title: ", "")
            chunks.append({
                "id": chunk_id,
                "blocks": blocks,
                "tokens": tokens,
                "title": title if title != "(none)" else None,
            })
    return chunks


def evaluate_chunking(aif_file: str, strategy: str, max_tokens: int = MAX_TOKENS) -> dict:
    """Evaluate a chunking strategy on a single document. Returns metrics dict."""
    chunks = chunk_document(aif_file, strategy, max_tokens)
    if not chunks:
        return {"error": "no chunks produced", "file": os.path.basename(aif_file), "strategy": strategy}

    n = len(chunks)
    tokens_list = [c["tokens"] for c in chunks]
    blocks_list = [c["blocks"] for c in chunks]

    # Self-containment: fraction of chunks with a title
    titled = sum(1 for c in chunks if c["title"])
    self_containment = titled / n if n > 0 else 0

    # Token budget compliance (token-budget strategy only)
    if strategy == "token-budget":
        within_budget = sum(1 for t in tokens_list if t <= max_tokens * 1.1)
        budget_compliance = within_budget / n if n > 0 else 0
    else:
        budget_compliance = None

    # Size coefficient of variation
    mean_tokens = sum(tokens_list) / n if n > 0 else 0
    if mean_tokens > 0 and n > 1:
        variance = sum((t - mean_tokens) ** 2 for t in tokens_list) / (n - 1)
        cv = math.sqrt(variance) / mean_tokens
    else:
        cv = 0.0

    # Average blocks per chunk
    avg_blocks = sum(blocks_list) / n if n > 0 else 0

    return {
        "file": os.path.basename(aif_file),
        "strategy": strategy,
        "num_chunks": n,
        "total_tokens": sum(tokens_list),
        "mean_tokens": round(mean_tokens, 1),
        "min_tokens": min(tokens_list) if tokens_list else 0,
        "max_tokens": max(tokens_list) if tokens_list else 0,
        "self_containment": round(self_containment, 3),
        "budget_compliance": round(budget_compliance, 3) if budget_compliance is not None else None,
        "size_cv": round(cv, 3),
        "avg_blocks_per_chunk": round(avg_blocks, 1),
    }

## Run Benchmark

Run all 4 strategies against every discovered `.aif` file. This is deterministic and requires no API key.

In [ ]:
all_results = []
file_count = len(aif_files)
total_runs = file_count * len(STRATEGIES)
completed = 0

for aif_file in aif_files:
    rel = aif_file.relative_to(PROJECT_ROOT)
    print(f"\n--- {rel} ---")
    for strategy in STRATEGIES:
        result = evaluate_chunking(str(aif_file), strategy)
        all_results.append(result)
        completed += 1
        if "error" in result:
            print(f"  [{completed}/{total_runs}] {strategy:15s}  ERROR: {result['error']}")
        else:
            sc_pct = f"{result['self_containment']:.0%}"
            print(
                f"  [{completed}/{total_runs}] {strategy:15s}  "
                f"chunks={result['num_chunks']:2d}  "
                f"avg_tok={result['mean_tokens']:7.1f}  "
                f"self_cont={sc_pct:>4s}  "
                f"cv={result['size_cv']:.2f}"
            )

print(f"\nDone. {len(all_results)} results collected.")

## Results Table

Aggregate metrics per strategy across all documents. The key question: which strategy produces the most **self-contained** chunks (good for LLM reasoning) with the most **uniform sizes** (good for context window packing)?

In [ ]:
print(f"{'Strategy':15s} {'Avg Chunks':>11s} {'Self-Cont.':>11s} {'Size CV':>8s} {'Avg Blk/Chk':>13s}")
print("-" * 62)

strategy_summaries = {}

for strategy in STRATEGIES:
    strat_results = [r for r in all_results if r.get("strategy") == strategy and "error" not in r]
    if not strat_results:
        continue

    total_chunks = sum(r["num_chunks"] for r in strat_results)
    avg_chunks = total_chunks / len(strat_results)
    avg_sc = statistics.mean(r["self_containment"] for r in strat_results)
    avg_cv = statistics.mean(r["size_cv"] for r in strat_results)
    avg_bpc = statistics.mean(r["avg_blocks_per_chunk"] for r in strat_results)

    strategy_summaries[strategy] = {
        "avg_chunks": round(avg_chunks, 1),
        "self_containment": round(avg_sc, 3),
        "size_cv": round(avg_cv, 3),
        "avg_blocks_per_chunk": round(avg_bpc, 1),
    }

    sc_pct = f"{avg_sc:.0%}"
    print(f"{strategy:15s} {avg_chunks:11.1f} {sc_pct:>11s} {avg_cv:8.2f} {avg_bpc:13.1f}")

# Highlight best strategy for self-containment
if strategy_summaries:
    best_sc = max(strategy_summaries, key=lambda s: strategy_summaries[s]["self_containment"])
    best_cv = min(strategy_summaries, key=lambda s: strategy_summaries[s]["size_cv"])
    print(f"\nHighest self-containment: {best_sc} ({strategy_summaries[best_sc]['self_containment']:.0%})")
    print(f"Most uniform chunk sizes: {best_cv} (CV={strategy_summaries[best_cv]['size_cv']:.2f})")

## Per-Document Details

Breakdown showing how each strategy performs on each individual document.

In [ ]:
# Group results by file
files_seen = []
for r in all_results:
    if r["file"] not in files_seen:
        files_seen.append(r["file"])

for filename in files_seen:
    file_results = [r for r in all_results if r["file"] == filename]
    print(f"\n{'=' * 70}")
    print(f"  {filename}")
    print(f"{'=' * 70}")
    print(f"  {'Strategy':15s} {'Chunks':>7s} {'Avg Tok':>8s} {'Range':>15s} {'Self-Cont':>10s} {'CV':>6s} {'Blk/Chk':>8s}")
    print(f"  {'-' * 69}")
    for r in file_results:
        if "error" in r:
            print(f"  {r['strategy']:15s}  ERROR: {r['error']}")
            continue
        tok_range = f"[{r['min_tokens']}-{r['max_tokens']}]"
        sc_pct = f"{r['self_containment']:.0%}"
        budget_note = ""
        if r["budget_compliance"] is not None:
            budget_note = f"  budget={r['budget_compliance']:.0%}"
        print(
            f"  {r['strategy']:15s} {r['num_chunks']:7d} {r['mean_tokens']:8.1f} {tok_range:>15s} "
            f"{sc_pct:>10s} {r['size_cv']:6.2f} {r['avg_blocks_per_chunk']:8.1f}{budget_note}"
        )

## Save Results

In [ ]:
output_path = Path(os.path.abspath("")) / "results.json"
output_data = {
    "strategy_summaries": strategy_summaries,
    "per_document": all_results,
}
with open(output_path, "w") as f:
    json.dump(output_data, f, indent=2)
print(f"Results saved to {output_path}")

## Findings

### Which strategy produces the most self-contained chunks?

**Section-based chunking** and **semantic chunking** tend to produce the highest self-containment scores. This is because they split at natural document boundaries --- section headings and semantic block boundaries --- so each chunk begins with clear context (a title or a typed block like `@step` or `@claim`).

Self-contained chunks matter because an LLM receiving a chunk in a retrieval pipeline needs to understand *what the chunk is about* without seeing the rest of the document. A chunk that starts with a section title or a clearly typed block (e.g., `@precondition: ...`) gives the model an immediate frame for reasoning.

### Which strategy is best for LLM consumption?

There is no single best strategy --- it depends on the use case:

- **For instruction-following agents** (e.g., executing skill steps): **Semantic** chunking preserves typed block boundaries. An agent receiving a `@step` + `@verify` pair in one chunk can execute and validate without cross-chunk lookups.
- **For context-window packing** (e.g., RAG with a fixed window): **Token Budget** guarantees no chunk exceeds the target, so you can pack exactly N chunks into a prompt without overflow.
- **For document comprehension** (e.g., summarization): **Section** chunking maps chunks to logical topics, which helps the LLM produce structured summaries.
- **Fixed Blocks** is the simplest strategy but often produces chunks with poor self-containment and high size variance. It is useful as a baseline but rarely optimal.

### Key insight: semantic boundaries improve instruction following

AIF's typed blocks (`@claim`, `@evidence`, `@step`, `@verify`, `@precondition`) carry explicit meaning that chunking strategies can respect. When a chunk boundary falls between a `@step` and its `@verify`, the LLM loses the ability to check its own work. Semantic and section-based strategies avoid this by keeping related typed blocks together, directly improving the model's ability to follow multi-step instructions.